# Simulasi Pushdown Automata (PDA) – Pengenal Kurung Seimbang

**Mata Kuliah:** Teori Bahasa dan Automata  
**Studi kasus:** Pengenalan tanda kurung bersarang menggunakan stack  
**Bahasa:** Python

Notebook ini mengikuti model formal pada Makalah Kelompok 10. Setiap `(` menambahkan simbol `X` ke stack (*push*), sedangkan setiap `)` yang sah menghapus `X` teratas (*pop*). Input diterima jika seluruh karakter telah dibaca dan mesin mencapai state final `qf` dengan stack menyisakan simbol dasar `Z0`.

> Kode ini adalah simulator edukatif buatan untuk menelusuri model PDA pada makalah, bukan library eksternal atau parser produksi.

## 1. Definisi formal PDA

\[
M = (Q, \Sigma, \Gamma, \delta, q_0, Z_0, F)
\]

| Komponen | Nilai |
|---|---|
| `Q` | `{q, qf}` |
| `Σ` | `{(, )}` |
| `Γ` | `{X, Z0}` |
| `q0` | `q` |
| `Z0` | Simbol dasar stack |
| `F` | `{qf}` |

**Fungsi transisi:**
1. `δ(q, (, Z0) = (q, XZ0)` — membaca `(` dan melakukan push `X`.
2. `δ(q, (, X) = (q, XX)` — membaca `(` dan melakukan push `X`.
3. `δ(q, ), X) = (q, ε)` — membaca `)` dan melakukan pop `X`.
4. `δ(q, ε, Z0) = (qf, Z0)` — berpindah ke state final tanpa mengonsumsi input.

Transisi ε dapat dipilih ketika stack kembali ke `Z0`. Penerimaan tetap mensyaratkan input habis. Karena ada pilihan antara membaca input berikutnya dan mengambil transisi ε, model ini bersifat nondeterministik.

## 2. Implementasi simulator NPDA

Stack disimpan sebagai tuple dari dasar ke puncak. Penelusuran konfigurasi menggunakan BFS untuk memeriksa cabang transisi yang membaca input maupun transisi ε.

In [1]:
from collections import deque

STATE_START = "q"
STATE_FINAL = "qf"
BOTTOM = "Z0"
STACK_MARKER = "X"


def simulate_pda(input_string):
    """Simulasi NPDA untuk bahasa tanda kurung seimbang."""
    # Konfigurasi: (state, posisi input, stack, jejak)
    queue = deque([(STATE_START, 0, (BOTTOM,), [])])
    visited = set()

    while queue:
        state, pos, stack, trace = queue.popleft()
        key = (state, pos, stack)
        if key in visited:
            continue
        visited.add(key)

        remaining = input_string[pos:] or "ε"
        current_trace = trace + [{
            "state": state,
            "remaining_input": remaining,
            "stack": " ".join(stack)
        }]

        # Input harus habis ketika mencapai state final.
        if state == STATE_FINAL and pos == len(input_string):
            return True, current_trace

        if state != STATE_START:
            continue

        top = stack[-1]

        # Membaca '(' : push X.
        if pos < len(input_string) and input_string[pos] == "(":
            queue.append((STATE_START, pos + 1,
                          stack + (STACK_MARKER,), current_trace))

        # Membaca ')' : pop X. Z0 tidak boleh di-pop.
        if (pos < len(input_string) and input_string[pos] == ")"
                and top == STACK_MARKER):
            queue.append((STATE_START, pos + 1,
                          stack[:-1], current_trace))

        # Transisi epsilon menuju qf ketika stack kembali ke Z0.
        if top == BOTTOM:
            queue.append((STATE_FINAL, pos, stack, current_trace))

    return False, []


def show_simulation(input_string):
    accepted, trace = simulate_pda(input_string)
    print(f"Input: {input_string!r}")
    print("Hasil:", "DITERIMA" if accepted else "DITOLAK")
    if trace:
        print("\\nJejak konfigurasi:")
        for i, item in enumerate(trace):
            print(f"{i}. state={item['state']} | "
                  f"sisa input={item['remaining_input']} | "
                  f"stack={item['stack']}")
    return accepted


## 3. Contoh input diterima
Contoh ini menunjukkan kurung bersarang `(())` dan dua pasangan berdampingan `()()`.

In [2]:
show_simulation("(())")

Input: '(())'
Hasil: DITERIMA
\nJejak konfigurasi:
0. state=q | sisa input=(()) | stack=Z0
1. state=q | sisa input=()) | stack=Z0 X
2. state=q | sisa input=)) | stack=Z0 X X
3. state=q | sisa input=) | stack=Z0 X
4. state=q | sisa input=ε | stack=Z0
5. state=qf | sisa input=ε | stack=Z0


True

In [3]:
show_simulation("()()")

Input: '()()'
Hasil: DITERIMA
\nJejak konfigurasi:
0. state=q | sisa input=()() | stack=Z0
1. state=q | sisa input=)() | stack=Z0 X
2. state=q | sisa input=() | stack=Z0
3. state=q | sisa input=) | stack=Z0 X
4. state=q | sisa input=ε | stack=Z0
5. state=qf | sisa input=ε | stack=Z0


True

## 4. Pengujian beberapa input
Input ditolak jika penutup tidak memiliki pembuka yang tersedia atau masih ada pembuka yang belum ditutup ketika input habis.

In [4]:
test_cases = ["()", "(())", "()()", "(()())", "", ")(", "())(", "(()", "((()))"]

for value in test_cases:
    result, _ = simulate_pda(value)
    print(f"{value!r:10} -> {'DITERIMA' if result else 'DITOLAK'}")

'()'       -> DITERIMA
'(())'     -> DITERIMA
'()()'     -> DITERIMA
'(()())'   -> DITERIMA
''         -> DITERIMA
')('       -> DITOLAK
'())('     -> DITOLAK
'(()'      -> DITOLAK
'((()))'   -> DITERIMA


## 5. Analisis dan pemetaan konsep ke kode

| Konsep PDA | Implementasi Python |
|---|---|
| State awal `q` | `STATE_START` |
| State final `qf` | `STATE_FINAL` |
| Simbol dasar `Z0` | `BOTTOM` |
| Penanda stack `X` | `STACK_MARKER` |
| Posisi kepala baca | `pos` |
| Memori stack | `stack` |
| Push | `stack + (STACK_MARKER,)` |
| Pop | `stack[:-1]` |
| Transisi ε | Menambahkan konfigurasi `qf` tanpa mengubah `pos` |
| Penerimaan | State `qf` dan posisi input sudah di akhir |

**Kesimpulan:** Stack mencatat jumlah tanda pembuka yang belum ditutup. Setiap `(` menambah satu `X`, sedangkan `)` yang sah menghapus satu `X`. Dengan demikian, model mengenali pasangan kurung seimbang termasuk struktur bersarang.

**Kompleksitas:** Untuk pengenal kurung khusus ini, pemrosesan satu lintasan membutuhkan waktu `O(n)` dan ruang stack terburuk `O(n)`, dengan `n` panjang input. Ini bukan klaim kompleksitas umum semua NPDA atau parser.

**Batasan:** Model hanya memeriksa satu jenis tanda kurung. Model belum melakukan tokenisasi, pemeriksaan nama tag HTML/XML, atribut, literal string, atau validasi semantik program. Sistem nyata memerlukan lexer dan parser yang lebih lengkap.

**Catatan pendekatan:** Kode merupakan simulator edukatif yang merepresentasikan transisi NPDA dalam makalah, bukan implementasi library eksternal.